In [38]:
! pip install pytorch-forecasting pytorch-lightning


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [39]:
import warnings
warnings.filterwarnings("ignore")

import os
import pandas as pd
import numpy as np
import torch
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping
from lightning.pytorch.tuner import Tuner
import matplotlib.pyplot as plt

from pytorch_forecasting import Baseline, DeepAR, TimeSeriesDataSet
from pytorch_forecasting.data import NaNLabelEncoder
from pytorch_forecasting.metrics import MAE, SMAPE
from sklearn.model_selection import train_test_split

max_encoder_length = 128   # Context window length
max_prediction_length = 32 # Forecast horizon

# Choose which target variable to forecast (e.g., "ENGINE_RPM ()")
target = "ENGINE_RPM ()"
time_col = 'ENGINE_RUN_TINE ()'

# Data loading
path = "/Users/darenpalmer/Desktop/UCL/CS/fyp.nosync/data/carOBD/obdiidata" 

df_list = []
for file in os.listdir(path):
    if file.endswith('.csv'):
        df = pd.read_csv(f'{path}/{file}', index_col=False)
        df['drive_id'] = file
        df_list.append(df)

print(f'{len(df_list)} files loaded out of {len([f for f in os.listdir(path) if f.endswith(".csv")])}')


129 files loaded out of 129


In [40]:
def remove_zero_variance_columns(df: pd.DataFrame, exclude_cols: list[str] = None) -> pd.DataFrame:
    """
    Compute std of each std-computable column (numeric only)
    """
    if exclude_cols is None:
        exclude_cols = []
    
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    cols_to_check = [col for col in numeric_cols if col not in exclude_cols]
  
    std_df = df[cols_to_check].std()
    zero_variance_cols = std_df[std_df == 0].index.tolist()
  
    print(f'{len(zero_variance_cols)} columns with zero variance: {zero_variance_cols}')
  
    if len(zero_variance_cols) > 0:
        df = df.drop(columns=zero_variance_cols)
  
    return df


In [41]:
def mean_fill_missing_timestamps_and_remove_duplicates(df: pd.DataFrame, time_col: str, id_cols: list[str] = None) -> pd.DataFrame:
    """
    Remove duplicate timestamps by averaging all numeric columns for each unique timestamp.
    This preserves the overall statistics while removing duplicate entries.
    
    Note: The time column itself is not averaged (it becomes the group key).
    Only numeric columns are averaged when multiple rows share the same timestamp.
    """
    if id_cols is None:
        id_cols = []
    
    existing_id_cols = [col for col in id_cols if col in df.columns]
    
    group_cols = [time_col] + existing_id_cols
    
    agg_dict = {}
    for col in df.columns:
        if col not in group_cols:
            if pd.api.types.is_numeric_dtype(df[col]):
                agg_dict[col] = 'mean'
            else:
                agg_dict[col] = 'first'
  
    df_clean = df.groupby(group_cols, as_index=False).agg(agg_dict)
  
    return df_clean


In [42]:
def downsample(df, time_col, source_file_col, downsample_factor=2):
    result_dfs = []
    
    for source_file in df[source_file_col].unique():
        file_df = df[df[source_file_col] == source_file].copy()
        
        if len(file_df) < downsample_factor * 2:
            continue
        
        file_df = file_df.sort_values(time_col).reset_index(drop=True)
        
        # Simple decimation without pre-smoothing
        downsampled = file_df.iloc[::downsample_factor].copy()
        downsampled[time_col] = np.arange(len(downsampled)) * downsample_factor
        
        result_dfs.append(downsampled.reset_index(drop=True))
    
    return pd.concat(result_dfs, ignore_index=True)


In [43]:
def filter_long_drives(df, id_col='drive_id', min_length=160):
    """Keep only drives long enough for your context window"""
    drive_lengths = df.groupby(id_col).size()
    valid_drives = drive_lengths[drive_lengths >= min_length].index
    
    print(f"Keeping {len(valid_drives)}/{df[id_col].nunique()} drives")
    print(f"Dropped {len(df) - df[df[id_col].isin(valid_drives)].shape[0]} timesteps")
    
    return df[df[id_col].isin(valid_drives)].reset_index(drop=True)


In [44]:
# Combine all dataframes
data = pd.concat(df_list, ignore_index=True)

# Clean up
print(f"Total samples: {len(data):,}")
print(f"Unique drives: {data['drive_id'].nunique()}")

data = mean_fill_missing_timestamps_and_remove_duplicates(data, time_col=time_col, id_cols=["drive_id"])
data = remove_zero_variance_columns(data, exclude_cols=["drive_id"])
data = downsample(
    data,
    time_col=time_col,
    source_file_col='drive_id',
    downsample_factor=1
)

data = filter_long_drives(data, min_length=max_encoder_length + max_prediction_length)

# Add derivative features
data['speed_change'] = data.groupby('drive_id')['VEHICLE_SPEED ()'].diff()
data['rpm_change'] = data.groupby('drive_id')['ENGINE_RPM ()'].diff()


Total samples: 304,299
Unique drives: 129
3 columns with zero variance: ['FUEL_AIR_COMMANDED_EQUIV_RATIO ()', 'TIME_RUN_WITH_MIL_ON ()', 'DISTANCE_TRAVELED_WITH_MIL_ON ()']
Keeping 128/129 drives
Dropped 150 timesteps


In [45]:
def add_cross_channel_features(data, target_columns):
    """
    Engineer features that capture cross-channel relationships.
    Add these as conditional columns.
    """
    # RPM-to-Speed ratio (gear indicator)
    if 'ENGINE_RPM ()' in data.columns and 'VEHICLE_SPEED ()' in data.columns:
        data['RPM_SPEED_RATIO'] = data['ENGINE_RPM ()'] / (data['VEHICLE_SPEED ()'] + 1)
    
    # Throttle-to-Load ratio (efficiency indicator)
    if 'THROTTLE ()' in data.columns and 'ENGINE_LOAD ()' in data.columns:
        data['THROTTLE_LOAD_RATIO'] = data['THROTTLE ()'] / (data['ENGINE_LOAD ()'] + 1)
    
    # Speed-based categories
    if 'VEHICLE_SPEED ()' in data.columns:
        data['IS_IDLE'] = (data['VEHICLE_SPEED ()'] < 5).astype(float)
        data['IS_HIGHWAY'] = (data['VEHICLE_SPEED ()'] > 60).astype(float)
    
    # RPM acceleration
    if 'ENGINE_RPM ()' in data.columns:
        data['RPM_ACCEL'] = data.groupby('drive_id')['ENGINE_RPM ()'].diff().fillna(0)
    
    return data

# Apply cross-channel features before preprocessing
target_columns = [
    'COOLANT_TEMPERATURE ()',
    'ENGINE_RPM ()',
    'VEHICLE_SPEED ()',
    'THROTTLE ()',
    'ENGINE_LOAD ()',
    'INTAKE_MANIFOLD_PRESSURE ()',
]

data = add_cross_channel_features(data, target_columns)
print("Added cross-channel features")


Added cross-channel features


In [46]:
# Prepare time index for pytorch-forecasting
# Reset time index per drive_id to start from 0
data = data.sort_values(['drive_id', time_col]).reset_index(drop=True)
data['time_idx'] = data.groupby('drive_id').cumcount()

# Convert drive_id to string for categorical encoding
data = data.astype(dict(drive_id=str))

# Split data into train/validation/test by drive_id
# Use time-based split similar to documentation
training_cutoff = data["time_idx"].max() - max_prediction_length

train_data = data[data["time_idx"] <= training_cutoff].copy()
valid_data = data[data["time_idx"] > training_cutoff].copy()

print(f"Training samples: {len(train_data):,}, Validation samples: {len(valid_data):,}")
print(f"Training drives: {train_data['drive_id'].nunique()}, Validation drives: {valid_data['drive_id'].nunique()}")


Training samples: 82,225, Validation samples: 32
Training drives: 128, Validation drives: 1


In [50]:
train_data

,ENGINE_RUN_TINE (),drive_id,ENGINE_RPM (),VEHICLE_SPEED (),THROTTLE (),ENGINE_LOAD (),COOLANT_TEMPERATURE (),LONG_TERM_FUEL_TRIM_BANK_1 (),SHORT_TERM_FUEL_TRIM_BANK_1 (),INTAKE_MANIFOLD_PRESSURE (),...,TIME_SINCE_TROUBLE_CODES_CLEARED (),WARM_UPS_SINCE_CODES_CLEARED (),speed_change,rpm_change,RPM_SPEED_RATIO,THROTTLE_LOAD_RATIO,IS_IDLE,IS_HIGHWAY,RPM_ACCEL,time_idx
0,0,drive1.csv,232.96875,0.0,17.450980,37.058823,81.00,-4.101562,0.000000,85.625,...,8260.0,255.0,NaN,NaN,232.968750,0.458527,1.0,0.0,0.00000,0
1,1,drive1.csv,1124.31250,0.0,19.215687,30.882354,80.75,0.000000,0.000000,32.250,...,8260.0,255.0,0.0,891.34375,1124.312500,0.602706,1.0,0.0,891.34375,1
2,2,drive1.csv,1113.25000,0.0,19.215687,31.764706,80.00,0.000000,0.000000,33.000,...,8260.0,255.0,0.0,-11.06250,1113.250000,0.586475,1.0,0.0,-11.06250,2
3,3,drive1.csv,922.50000,0.0,18.823530,35.392156,79.75,-0.781250,0.000000,35.000,...,8260.0,255.0,0.0,-190.75000,922.500000,0.517241,1.0,0.0,-190.75000,3
4,4,drive1.csv,847.56250,0.0,18.823530,35.882354,79.00,-0.781250,0.000000,35.250,...,8260.0,255.0,0.0,-74.93750,847.562500,0.510367,1.0,0.0,-74.93750,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82252,507,ufpe9.csv,873.25000,17.0,16.470589,23.006536,87.00,0.000000,9.375000,24.000,...,13084.0,255.0,-1.5,-287.25000,48.513889,0.686088,0.0,0.0,-287.25000,507
82253,508,ufpe9.csv,858.37500,14.0,16.470589,23.823529,87.00,0.000000,5.273438,25.000,...,13084.0,255.0,-3.0,-14.87500,57.225000,0.663507,0.0,0.0,-14.87500,508
82254,509,ufpe9.csv,824.81250,10.0,16.470589,24.901961,87.00,0.000000,6.054688,25.250,...,13084.0,255.0,-4.0,-33.56250,74.982955,0.635882,0.0,0.0,-33.56250,509
82255,510,ufpe9.csv,808.50000,7.0,16.470589,25.490196,87.00,0.000000,3.125000,26.000,...,13084.0,255.0,-3.0,-16.31250,101.062500,0.621762,0.0,0.0,-16.31250,510


In [47]:
# Identify feature columns (exclude metadata columns)
exclude_cols = ['drive_id', time_col, 'time_idx', target]
feature_cols = [col for col in data.columns if col not in exclude_cols and col not in ['WARM_UPS_SINCE_CODES_CLEARED ()', 'TIME_SINCE_TROUBLE_CODES_CLEARED ()']]

# Create TimeSeriesDataSet following DeepAR documentation pattern
training = TimeSeriesDataSet(
    train_data,
    time_idx="time_idx",
    target=target,
    categorical_encoders={"drive_id": NaNLabelEncoder().fit(data.drive_id)},
    group_ids=["drive_id"],
    static_categoricals=["drive_id"],  # Important for forecasting correlations
    time_varying_unknown_reals=[target],  # Target variable
    time_varying_known_reals=feature_cols if len(feature_cols) > 0 else None,  # Known features
    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,
)

# Create validation dataset from training dataset
validation = TimeSeriesDataSet.from_dataset(
    training, 
    valid_data, 
    min_prediction_idx=training_cutoff + 1
)



ValueError: 128 (0.16%) of speed_change values were found to be NA or infinite (even after encoding). NA values are not allowed `allow_missing_timesteps` refers to missing rows, not to missing values. Possible strategies to fix the issue are (a) dropping the variable speed_change, (b) using `NaNLabelEncoder(add_nan=True)` for categorical variables, (c) filling missing values and/or (d) optionally adding a variable indicating filled values

In [ ]:
# Create data loaders with synchronized batch sampler (required for DeepAR)
batch_size = 64
train_dataloader = training.to_dataloader(
    train=True, 
    batch_size=batch_size, 
    num_workers=0, 
    batch_sampler="synchronized"  # Important for DeepAR
)
val_dataloader = validation.to_dataloader(
    train=False, 
    batch_size=batch_size, 
    num_workers=0, 
    batch_sampler="synchronized"  # Important for DeepAR
)


In [ ]:
# Calculate baseline error for comparison
baseline_predictions = Baseline().predict(
    val_dataloader, 
    trainer_kwargs=dict(accelerator="auto"), 
    return_y=True
)
baseline_mae = MAE()(baseline_predictions.output, baseline_predictions.y)
baseline_smape = SMAPE()(baseline_predictions.output, baseline_predictions.y)
print(f"Baseline MAE: {baseline_mae:.4f}")
print(f"Baseline SMAPE: {baseline_smape:.4f}")


In [ ]:
# Initialize trainer for learning rate finding
pl.seed_everything(42)
trainer = pl.Trainer(accelerator="auto", gradient_clip_val=0.1)

# Create DeepAR model
net = DeepAR.from_dataset(
    training,
    learning_rate=3e-2,  # Initial learning rate (will be tuned)
    hidden_size=64,
    rnn_layers=2,
    loss=MSE(),
    optimizer="Adam",
)


In [ ]:
# Find optimal learning rate using PyTorch Lightning Tuner
print("Finding optimal learning rate...")
res = Tuner(trainer).lr_find(
    net,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader,
    min_lr=1e-5,
    max_lr=1e0,
    early_stop_threshold=100,
)
suggested_lr = res.suggestion()
print(f"Suggested learning rate: {suggested_lr}")
fig = res.plot(show=True, suggest=True)
plt.show()
net.hparams.learning_rate = suggested_lr


In [ ]:
# Load best model from checkpoint
best_model_path = trainer.checkpoint_callback.best_model_path
best_model = DeepAR.load_from_checkpoint(best_model_path)
print(f"Loaded best model from: {best_model_path}")

# Evaluate on validation set
predictions = best_model.predict(
    val_dataloader, 
    trainer_kwargs=dict(accelerator="auto"), 
    return_y=True
)
val_mae = MAE()(predictions.output, predictions.y)
val_smape = SMAPE()(predictions.output, predictions.y)
print(f"Validation MAE: {val_mae:.4f}")
print(f"Validation SMAPE: {val_smape:.4f}")


In [ ]:
# Get raw predictions for plotting
raw_predictions = best_model.predict(
    val_dataloader,
    mode="raw",
    return_x=True,
    n_samples=100,
    trainer_kwargs=dict(accelerator="auto"),
)

# Plot predictions for multiple examples
series = validation.x_to_index(raw_predictions.x)["drive_id"]
for idx in range(min(10, len(raw_predictions.x))):  # Plot 10 examples
    best_model.plot_prediction(
        raw_predictions.x, 
        raw_predictions.output, 
        idx=idx, 
        add_loss_to_title=True
    )
    plt.suptitle(f"Drive ID: {series.iloc[idx]}")
    plt.show()


In [ ]:
# Save the trained model
torch.save(best_model.state_dict(), "sensor_forecasting_deepar.pth")
print("Model saved to sensor_forecasting_deepar.pth")


In [ ]:
# Set up early stopping callback
early_stop_callback = EarlyStopping(
    monitor="val_loss", 
    min_delta=1e-4, 
    patience=10, 
    verbose=False, 
    mode="min"
)

# Create trainer with early stopping
trainer = pl.Trainer(
    max_epochs=50,
    accelerator="auto",
    enable_model_summary=True,
    gradient_clip_val=0.1,
    callbacks=[early_stop_callback],
    enable_checkpointing=True,
)

# Create model with optimal learning rate
net = DeepAR.from_dataset(
    training,
    learning_rate=suggested_lr,
    log_interval=10,
    log_val_interval=1,
    hidden_size=64,
    rnn_layers=2,
    optimizer="Adam",
    loss=MSE(),
)

# Train the model
print("Starting training...")
trainer.fit(
    net,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader,
)
